# Termometro, fase 2 corta, `smol17` (HuggingFaceTB/SmolLM2-1.7B-Instruct), notebook B2: le 12 corse da 20 passi
Impostazioni: **GPU T4 x2**, **Internet attivo**. Ingressi: codice congelato, codice della fase 2, partenze e l'output del notebook A2. Controlla che la previsione sia identica a quella su git, poi semi 101-106 sulla GPU 0 e 107-112 sulla GPU 1 (`termometro.py run ... 20`), poi `fase2_corta.py analyze` (verdetto al passo 10, riserva al 5) e le code dei passi 16-20. Tutto come in `PREMESSA_FASE2_MODIFICA`.

In [ ]:
# ambiente, codice congelato, modello base e partenza (uguale in A e in B)
import os, glob, json, hashlib, shutil, subprocess, sys, time
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.5.4', 'peft==0.18.0'], check=True)
W = '/kaggle/working'; OUT = f'{W}/smol17'; os.makedirs(f'{OUT}/log', exist_ok=True)
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
CODE = os.path.dirname(glob.glob('/kaggle/input/**/MANIFEST_CONGELAMENTO.txt', recursive=True)[0])
man = [l.split() for l in open(f'{CODE}/MANIFEST_CONGELAMENTO.txt', encoding='utf-8') if l.strip() and not l.startswith('#')]
bad = [n for h, n in man if sha(f'{CODE}/{n}') != h]
assert not bad, f'CODICE DIVERSO DAL CONGELAMENTO: {bad}'
print('codice identico al congelamento:', len(man), 'file;', open(f'{CODE}/MANIFEST_CONGELAMENTO.txt', encoding='utf-8').readline().strip())
from huggingface_hub import snapshot_download
snapshot_download('HuggingFaceTB/SmolLM2-1.7B-Instruct', local_dir=f'{W}/base')
assert sha(f'{W}/base/model.safetensors') == 'f55217be716b6a997b97b9d8d7eb6fad02e00858f5010ec24f64603c3a98a0e8', 'modello base diverso da casa'
SFT_IN = os.path.dirname(glob.glob('/kaggle/input/**/smol17/sft/adapter_model.safetensors', recursive=True)[0])
shutil.copytree(SFT_IN, f'{OUT}/sft', dirs_exist_ok=True)
assert sha(f'{OUT}/sft/adapter_model.safetensors') == 'd158d49681ca2a78d79d2e314726f779f5f45bdc23bda9de0c695d57a9748f8a', 'partenza diversa da quella del 20/10'
cfg = json.load(open(f'{OUT}/sft/adapter_config.json')); cfg['base_model_name_or_path'] = f'{W}/base'
json.dump(cfg, open(f'{OUT}/sft/adapter_config.json', 'w'), indent=2)
import torch, transformers, peft
print('torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__, '| float32 (TERMO_FP32=1)')
def lancia(args, log, gpu):
    env = dict(os.environ, TERMO_FP32='1', CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED='1')
    return subprocess.Popen([sys.executable, '-X', 'utf8', f'{CODE}/termometro.py'] + args, stdout=open(log, 'w'),
                            stderr=subprocess.STDOUT, env=env, cwd=W)
def coda(log, n=3):
    return '\n'.join(open(log, encoding='utf-8', errors='replace').read().splitlines()[-n:])


In [ ]:
# codice della fase 2 corta (fase2_corta.py e analisi_fase2.py), con le impronte del congelamento
F2 = os.path.dirname(glob.glob('/kaggle/input/**/MANIFEST_FASE2.txt', recursive=True)[0])
m2 = [l.split() for l in open(f'{F2}/MANIFEST_FASE2.txt', encoding='utf-8') if l.strip() and not l.startswith('#')]
assert m2 and all(sha(f'{F2}/{n}') == h for h, n in m2), 'CODICE DELLA FASE 2 DIVERSO DAL CONGELAMENTO'
print('codice della fase 2 identico al congelamento:', open(f'{F2}/MANIFEST_FASE2.txt', encoding='utf-8').readline().strip())


In [ ]:
# la previsione: identica a quella su git (impronte copiate qui dal commit, prima di lanciare B)
GIT_SHA = {}  # <- da riempire con il contenuto di previsione_sha256.txt del commit su git
A = os.path.dirname(glob.glob('/kaggle/input/**/smol17/previsione_sha256.txt', recursive=True)[0])
for n in ['thermo_s1.json', 'thermo_s2.json', 'gates.json', 'gates_pool.json', 'pool.json', 'vettori_s1.pt', 'gradienti_s1.pt']:
    h = sha(f'{A}/{n}')
    assert GIT_SHA and h == GIT_SHA[n], f'PREVISIONE DIVERSA DA QUELLA SU GIT: {n}'
    shutil.copy(f'{A}/{n}', f'{OUT}/{n}')
for f in glob.glob('/kaggle/input/**/smol17/run_s*.json', recursive=True):  # corse complete di una versione precedente
    if [r['step'] for r in json.load(open(f))] == list(range(21)):
        shutil.copy(f, OUT)
        c = f.replace('.json', '_code.jsonl')
        if os.path.exists(c):
            shutil.copy(c, OUT)
print('previsione identica a git; corse già complete:', sorted(os.path.basename(f) for f in glob.glob(f'{OUT}/run_s*.json')))


In [ ]:
# le corse: 6 per GPU, una alla volta per GPU, con il limite di tempo della sessione
T_START, LIMITE = time.time(), 11.0 * 3600
def completa(s):
    f = f'{OUT}/run_s{s}.json'
    try:  # complete = rows for steps 0..20 in order; a file cut by a stopped process is incomplete (as in 1b)
        return os.path.exists(f) and [r['step'] for r in json.load(open(f))] == list(range(21))
    except (json.JSONDecodeError, OSError, KeyError, TypeError):
        return False
code = {'0': [s for s in range(101, 107) if not completa(s)], '1': [s for s in range(107, 113) if not completa(s)]}
corrente, tentativi, durate = {}, {}, []
while any(code.values()) or corrente:
    for gpu in ('0', '1'):
        p = corrente.get(gpu)
        if p and p[1].poll() is not None:
            s, proc, t = p
            del corrente[gpu]
            if completa(s):
                durate.append(time.time() - t)
                print(f'corsa {s} finita in {round((time.time() - t) / 60)} min', flush=True)
            elif tentativi.get(s, 0) < 2:
                code[gpu].insert(0, s)
                print(f'corsa {s} fallita, la ripeto una volta', flush=True)
            else:
                print(f'corsa {s} fallita due volte: esclusa (si scrive nel rapporto)', flush=True)
        if gpu not in corrente and code[gpu]:
            stima = max(durate) if durate else 45 * 60
            if time.time() - T_START + stima > LIMITE:
                print(f'tempo quasi finito: la GPU {gpu} non lancia {code[gpu]}', flush=True)
                code[gpu] = []
                continue
            s = code[gpu].pop(0)
            tentativi[s] = tentativi.get(s, 0) + 1
            log = f'{OUT}/log/run_s{s}.log' if tentativi[s] == 1 else f'{OUT}/log/run_s{s}_ripetuta.log'
            corrente[gpu] = (s, lancia(['run', f'{OUT}/sft', f'{OUT}/pool.json', OUT, f'{OUT}/run_s{s}.json', str(s), '20'], log, gpu), time.time())
    time.sleep(60)
fatte = sorted(s for s in range(101, 113) if completa(s))
print('corse complete:', fatte)


In [ ]:
# analisi della fase 2 corta per questa famiglia e code dei passi 16-20 (solo con tutte le 12 corse)
if len(fatte) == 12:
    env = dict(os.environ, PYTHONPATH=CODE)  # fase2_corta.py imports the frozen analisi_fase1.py
    esiti = []
    for script, args in [(f'{F2}/fase2_corta.py', ['analyze', OUT, OUT, f'{OUT}/analisi_fase2_corta.json']),
                         (f'{CODE}/estrai_code_passo40.py', [OUT, f'{OUT}/g5_passo20'])]:
        r = subprocess.run([sys.executable, '-X', 'utf8', script] + args, capture_output=True, text=True, env=env)
        print(os.path.basename(script), r.returncode); print(r.stdout[-3000:]); print(r.stderr[-1500:])
        esiti.append(r.returncode)
    if esiti == [0, 0]:  # the marker only when both steps worked (review 21/10)
        with open(f'{OUT}/FASE2_smol17_FINITA.txt', 'w') as f:
            f.write(time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()) + '\n')
    else:
        print('ANALISI O CODE FALLITE: niente segnale di fine; le corse sono comunque tra gli output')
else:
    print('mancano corse: nuova versione con questo output come ingresso')
shutil.rmtree(f'{W}/base', ignore_errors=True)
